# TCGA SARC data exploration
The repository includes the two input files in data/raw. After installing the dependencies, run `python -m src.data.prepare` from the repository root to generate data/processed. No separate download or extraction is needed. This notebook reads prepared data and calls the shared audit module; it does not implement another merge. See DATA_SOURCES.md for provenance and checksums.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "configs/data.yaml").exists():
    raise RuntimeError("Start Jupyter from the repository root or notebooks directory")
sys.path.insert(0, str(ROOT))
from src.data.audit import load_config, audit
config = load_config()
audit(config)

In [ ]:
dataset = pd.read_csv(config["processed_dir"] / "sarc_merged_six_subtypes.tsv", sep="\t")
genes = dataset.filter(regex="^gene__")
print(f"Samples: {len(dataset)}, patients: {dataset["Patient ID"].nunique()}, genes: {genes.shape[1]}")
dataset["subtype"].value_counts().rename_axis("Subtype").to_frame("Samples")

In [ ]:
pd.Series({"missing_expression_cells": int(genes.isna().sum().sum()),
           "zero_expression_fraction": float((genes == 0).to_numpy().mean()),
           "minimum_expression": float(genes.min().min()),
           "maximum_expression": float(genes.max().max())})

## Modeling boundary
Only gene__ columns are model inputs. subtype is the target; Patient ID is the validation group. Fit filtering, imputation, scaling, and feature selection within training folds. Keep this notebook descriptive.